# 04 - Lasso Regression para PM2.5

Este notebook implementa Lasso Regression usando solo variables numericas. No se usa `ESTACION` directamente porque es texto; Lasso necesita variables numericas para ajustar coeficientes.

## GUIA RAPIDA PARA EXPONER LASSO

Lasso Regression es un modelo lineal. En este proyecto se usa para estimar `PM 2.5` a partir de variables numericas.

Punto clave para explicar: Lasso no usa directamente `ESTACION` porque es texto. Para mantener el modelo simple e interpretable, aqui se trabaja solo con contaminantes y variables temporales numericas.

## 1. Librerias y rutas

In [ ]:
# Librerias, rutas y variables principales del notebook.
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'datos_modelo.csv'
RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'datos_horarios_contaminacion_lima.xlsx'
MODELS_DIR = PROJECT_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

TARGET = 'PM 2.5'
LASSO_FEATURES = ['PM 10', 'SO2', 'NO2', 'O3', 'CO', 'ANO', 'MES', 'DIA', 'HORA']

def clean_if_needed():
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)

    df = pd.read_excel(RAW_PATH)
    pollutants = ['PM 10', 'PM 2.5', 'SO2', 'NO2', 'O3', 'CO']
    for col in pollutants:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.drop_duplicates(subset=['CODIGO ESTACION', 'ANO', 'MES', 'DIA', 'HORA']).copy()
    df = df.sort_values(['CODIGO ESTACION', 'ANO', 'MES', 'DIA', 'HORA'])

    for col in ['PM 10', 'SO2', 'NO2', 'O3', 'CO']:
        df[col] = df.groupby('CODIGO ESTACION')[col].transform(lambda s: s.interpolate(limit_direction='both'))
        df[col] = df[col].fillna(df.groupby(['CODIGO ESTACION', 'HORA'])[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())

    df = df.dropna(subset=[TARGET] + LASSO_FEATURES).copy()
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df[LASSO_FEATURES + [TARGET]].to_csv(DATA_PATH, index=False)
    return df[LASSO_FEATURES + [TARGET]]


## 2. Variables usadas por Lasso

Lasso trabaja con coeficientes numericos. Por eso se usan contaminantes y variables de tiempo: `PM 10`, `SO2`, `NO2`, `O3`, `CO`, `ANO`, `MES`, `DIA`, `HORA`. La variable objetivo es `PM 2.5`.

### Comentario para exposicion: variables de Lasso

Variables de entrada (`X`): `PM 10`, `SO2`, `NO2`, `O3`, `CO`, `ANO`, `MES`, `DIA`, `HORA`.

Variable objetivo (`y`): `PM 2.5`.

La idea es que el modelo aprenda una relacion matematica entre esas variables numericas y el valor real de `PM 2.5`.

In [ ]:
df = clean_if_needed()
print(df.shape)
df[LASSO_FEATURES + [TARGET]].head()

## 3. Separacion de datos

### Comentario para exposicion: train/test

Se separan los datos en entrenamiento y prueba. El modelo aprende con entrenamiento y se evalua con prueba. Esto evita evaluar el modelo con los mismos datos que ya vio.

In [ ]:
# X son las variables de entrada; y es el PM 2.5 real que se quiere predecir.
X = df[LASSO_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print('Entrenamiento:', X_train.shape)
print('Prueba:', X_test.shape)

## 4. Pipeline de preprocesamiento y modelo

### Comentario para exposicion: pipeline de Lasso

El pipeline hace dos cosas: primero prepara los datos y luego entrena el modelo.

- `SimpleImputer`: rellena faltantes numericos con la mediana.
- `StandardScaler`: escala las variables para que Lasso compare columnas en magnitudes parecidas.
- `Lasso`: modelo lineal con regularizacion L1.

In [ ]:
# Pipeline: prepara las variables numericas y luego aplica Lasso Regression.
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), LASSO_FEATURES),
    ],
    verbose_feature_names_out=False,
)

lasso_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso(alpha=0.05, max_iter=10000, random_state=42)),
])
lasso_model

## 5. Entrenamiento

In [ ]:
lasso_model.fit(X_train, y_train)
y_pred = lasso_model.predict(X_test)
print('Modelo Lasso entrenado con variables numericas')

## 6. Evaluacion

### Comentario para exposicion: metricas

Aqui se calcula que tan bien predice Lasso.

- `MAE`: error absoluto promedio.
- `RMSE`: error en la misma unidad de PM 2.5. Menor es mejor.
- `R2`: que tanto explica el modelo. Mayor es mejor.

In [ ]:
# Calculo de metricas para medir el error del modelo.
mse = mean_squared_error(y_test, y_pred)
metrics_lasso = {
    'MAE': mean_absolute_error(y_test, y_pred),
    'MSE': mse,
    'RMSE': np.sqrt(mse),
    'R2': r2_score(y_test, y_pred),
}
pd.DataFrame([metrics_lasso])

## 7. Coeficientes

### Comentario para exposicion: coeficientes

Como Lasso es lineal, cada variable tiene un coeficiente. Un coeficiente mas grande en valor absoluto indica mayor influencia dentro del modelo, despues del escalado.

In [ ]:
feature_names = lasso_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = lasso_model.named_steps['model'].coef_
coef_df = pd.DataFrame({'Variable': feature_names, 'Coeficiente': coefficients})
coef_df['Valor absoluto'] = coef_df['Coeficiente'].abs()
coef_df.sort_values('Valor absoluto', ascending=False)

## 8. Grafico real vs predicho

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.25, s=8)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red')
plt.title('PM2.5 real vs predicho - Lasso Regression')
plt.xlabel('PM2.5 real')
plt.ylabel('PM2.5 predicho')
plt.show()

## 9. Guardado del modelo

### Comentario para exposicion: archivo .pkl

Al final se guarda el modelo entrenado en `models/lasso_regression.pkl`. Ese archivo permite que el dashboard use Lasso sin volver a entrenarlo cada vez.

In [ ]:
# Guardado del modelo entrenado para usarlo luego en el dashboard.
joblib.dump(lasso_model, MODELS_DIR / 'lasso_regression.pkl')
print(MODELS_DIR / 'lasso_regression.pkl')

## GRAFICOS GENERADOS POR LASSO

### Resultado visual del modelo Lasso Regression

Este grafico compara el valor real de `PM 2.5` contra el valor predicho por Lasso. Mientras mas cerca esten los puntos de la linea roja, mejor predice el modelo.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
IMAGE_DIR = PROJECT_DIR / 'reports' / 'imagenes'

print('Lasso Regression: PM2.5 real vs predicho')
display(Image(filename=str(IMAGE_DIR / 'lasso_real_vs_predicho.png')))